# Initialization

In [0]:
from pyspark.sql import functions as F

# Path Configuration

In [0]:
CATALOG = "humanoid_robotics"
MANIFEST_PATH = f"/Volumes/{CATALOG}/landing/manifest/manifest_pilot.csv"
RAW_DOCS_PATH = f"/Volumes/{CATALOG}/landing/raw_docs/"
TARGET_TABLE  = f"{CATALOG}.bronze.documents"

# Read manifest

In [0]:
df_manifest = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(MANIFEST_PATH)
)

df_manifest = df_manifest.toDF(*[c.replace("\ufeff", "").strip() for c in df_manifest.columns])

df_manifest.display()

# Read txt

In [0]:
df_docs = (
    spark.read
    .text(RAW_DOCS_PATH, wholetext=True)
    .select(
        F.col("value").alias("raw_text"),
        F.col("_metadata.file_path").alias("file_path"),
    )
    .withColumn("file_name", F.element_at(F.split("file_path", "/"), -1))  
)


df_docs.display()

# Join manifest and the text

In [0]:
df = (
    df_manifest
    .join(df_docs, on = "file_name", how = "left")
    .withColumn("publish_date", F.to_date("publish_date"))
    .withColumn("char_count", F.length("raw_text"))
    .withColumn("ingested_at", F.current_timestamp())
    .select("doc_id", "url", "title", "publisher", "source_type", "language", "publish_date", "primary_company", "file_name", "file_path","raw_text", "char_count", "ingested_at")
)
df.display()

# Sanity Check

In [0]:
total     = df.count()
missing   = df.filter(F.col("raw_text").isNull()).count()
too_short = df.filter(F.col("char_count") < 300).count()
dup_ids   = total - df.select("doc_id").distinct().count()

print(f"total_rows: {total}")
print(f"null_text: {missing}")
print(f"texts<300: {too_short}")
print(f"dup_doc_id: {dup_ids}")

display(df.filter("language = 'zh'").select("doc_id", F.substring("raw_text", 1, 200).alias("preview")))

# Ingest Files into bronze table

In [0]:
(
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", True)
          .saveAsTable(TARGET_TABLE)
)

In [0]:
%sql

select * from humanoid_robotics.bronze.documents